OpenPaths: Road Reallocation for Active Mobility Enhancement
===============

Matthew Pirotta

# Setup

Imports

In [ ]:
# Automatically reloads modules before each cell execution, so code changes in libraries are always propogated.
%load_ext autoreload
%autoreload 2

%load_ext snakeviz

import networkx as nx
from networkx import MultiDiGraph
import osmnx as ox
import numpy as np
import pandas as pd
import copy
import re
from tqdm.notebook import tqdm
import sumolib
import pickle
import json
import ast
import xml.etree.ElementTree as ET


from NetworkOptimisation import graph_optimisation_heuristic
from PreProcessing import graph_preprocessing, enrich_attributes, graph_structure
from NetworkOptimisation import heuristic as heuristic_optim
from Demand import ODGeneration, paths_util, sumoExporter, ODCalibrator, sumoRun, ODAggregation, ODConstants, SpatialPrep, ODSampling
import Plotting

import GraphUtil as graph_util
import sumo_util
import investigation
import constants

from constants import OD
from NetworkOptimisation import impedance_calculator

Config and Paths

In [ ]:
ox.settings.useful_tags_way = [
    "bridge", "tunnel", "oneway", "lanes", "highway",
    "width", "name", "name:en", "bicycle", "cycleway", "tunnel",
    "junction", "turns", "bike_cost_penalty" ]

LOCATION = "Malta (island)"
safe_location = re.sub(r"[()\[\]{}]", "", LOCATION)
safe_location = re.sub(r"[^0-9A-Za-z_-]+", "_", safe_location)
safe_location = re.sub(r"_+", "_", safe_location).strip("_")

SEED = 72
K_SAMPLE = 50
N_ITERATIONS = 20 

SUMO_OD_GENERATION_TIMELINE = [10_000, 20_000, 30_000, 40_000]
OD_GENERATION_TIMELINE = [1_000]

rng = np.random.default_rng(SEED)

nx.config.cache_converted_graphs = False

# Load/builg Graphs

In [ ]:
#NOTE the length attribute is given in meters
#You project not to fix lengths, but to make sure your geometry math (like areas or spatial intersections) makes sense in linear units.
#Simplify is used to significantly reduce the number of nodes, removing dead ends and combining nodes with no intersections
G_master_unsimplified = graph_preprocessing.load_network(LOCATION, simplify=False, truncate_largest_component=True, all_oneway=False)
G_master_unsimplified = graph_preprocessing.clean_unsimplified_graph(G_master_unsimplified)
ox.save_graph_xml(G_master_unsimplified, f"../Simulations/{safe_location}/{safe_location}.osm")

G_master_simplified = ox.simplify_graph(G_master_unsimplified, track_merged=True, edge_attrs_differ=["car_allowed"])
G_master_simplified, gdf_regions_proj, gdf_local_proj = graph_preprocessing.clean_simplified_graph(G_master_simplified, LOCATION)
print(f"Nodes:{len(G_master_simplified.nodes)}, Edges{len(G_master_simplified.edges)}")

G_plotting = graph_util.make_plotting_subgraph(G_master_simplified)

In [ ]:
G_master_unsimplified_oneway = graph_preprocessing.load_sumo_network(LOCATION, simplify=False,)
G_master_unsimplified_oneway = graph_preprocessing.clean_unsimplified_graph(G_master_unsimplified_oneway)

In [ ]:
ox.io.save_graphml(G_master_unsimplified_oneway, ) #GRAPH_PATH

ox.save_graph_xml(G_master_unsimplified_oneway, filepath=f"../Simulations/{safe_location}/{safe_location}.osm")

In [ ]:
arc_to_seg, seg_to_arcs = graph_util.build_arc_to_segment_map(G_master_simplified)

# Network Inspection

edge data

In [ ]:
edges = list(G_master_simplified.edges(keys=True, data=True))
u, v, k, data = edges[0]
print(u, v, k)
display(data)

Road type classification

In [ ]:
graph_util.summarise_road_type_stats(G_master_simplified)

graph node and edge count

In [ ]:
print("G_Master:")
print("Nodes:", G_master_simplified.number_of_nodes())
print("Edges (total, counts parallel edges separately):", G_master_simplified.number_of_edges())

print("-----")
print("G_Drive")
G_drive = graph_util.make_drive_subgraph(G_master_simplified)
print("Nodes:", G_drive.number_of_nodes())
print("Edges (total, counts parallel edges separately):", G_drive.number_of_edges())


print("-----")
print("G_bike")
G_bike = graph_util.make_bikeable_subgraph(G_master_simplified)
print("Nodes:", G_bike.number_of_nodes())
print("Edges (total, counts parallel edges separately):", G_bike.number_of_edges())

In [ ]:
fig, ax = Plotting.plot_categorical_attr(G_master_simplified, "safety", constants.SAFETY_COLORS,  fig_size=(10,10), dpi=500) # title = "Malta Car Road Safety Classification",
fig.savefig("Output/Plots/Malta_Road_Safety_Classification.pdf", format="pdf", bbox_inches="tight")

Parallel Edges

In [ ]:
investigation.analyze_parallel_edges(G_master_unsimplified_oneway)
investigation.compare_parallel_edges(G_master_unsimplified_oneway)
#investigation.plot_parallel_edges(G_master_unsimplified_oneway) 
print("-----")

investigation.analyze_parallel_edges(G_master_simplified)
investigation.compare_parallel_edges(G_master_simplified)
#investigation.plot_parallel_edges(G_master_simplified) 
print("-----")

investigation.analyze_parallel_edges(G_drive)
investigation.compare_parallel_edges(G_drive)
#investigation.plot_parallel_edges(G_drive)
print("-----")

## Reallocatability

In [ ]:
bridges = set(nx.bridges(G_master_simplified.to_undirected()))
print(f"No. of bridges: {len(bridges)}")

In [ ]:
nx.is_strongly_connected(G_drive)

In [ ]:
G_drive = graph_util.make_drive_subgraph(G_master_simplified)
Plotting.plot_boolean_attribute(G_drive, "reallocatable", fig_size=(40,40))

In [ ]:
reallocatable_false = 0
reallocatable_true =0
for u, v, k, d in G_drive.edges(keys=True, data=True):
    if d.get("reallocatable"):
        reallocatable_true += 1
    else:
        reallocatable_false += 1

print(f"realloacable true: {reallocatable_true},", f"realloaclable false {reallocatable_false},", f"{reallocatable_true/(reallocatable_true+reallocatable_false)*100}% are reallocable")

## Topology

In [ ]:
#Plotting.plot_elevation(G_master_simplified)
fig, ax = Plotting.plot_grades(G_master_simplified)


## Network statistics

# Generating city level subgraphs

In [ ]:
display(gdf_regions_proj.head())
display(gdf_local_proj.head())

fig, ax, _ = Plotting.plot_gdf_and_overlay(gdf_regions_proj, fontsize=12, annotate=True,  show=False) #title= "Regions",
fig.savefig("Output/Plots/Regions.pdf", format="pdf", bbox_inches="tight", pad_inches=0)


fig, ax, gdf_local_proj_coloured = Plotting.plot_gdf_and_overlay(gdf_local_proj, annotate=True, show=False) #title = "Localities",
fig.savefig("Output/Plots/Localities.pdf", format="pdf", bbox_inches="tight", pad_inches=0)


Plotting.plot_gdf_and_overlay(gdf_local_proj, G=G_master_simplified, annotate=False, show=False) #title="Localities + Roads",

Locality Level Subraphs

In [ ]:
subgraphs = {}
total_nodes = 0
total_edges = 0
for idx, row in gdf_local_proj.iterrows():
    name = row["name"]

    G_sub = graph_util.make_locality_subgraph(G_master_simplified, name)
    subgraphs[name] = G_sub
    total_nodes += len(G_sub.nodes)
    total_edges += len(G_sub.edges)
    print(f"✅ Created subgraph for {name}: {len(G_sub.nodes)} nodes, {len(G_sub.edges)} edges")

print(f"total_nodes{total_nodes}, total_edges{total_edges}")

# OD Matrix Generation

Origin and Destination valid sample region and points

In [ ]:
# https://wiki.openstreetmap.org/wiki/Key:amenity
gdf_destinations_raw = ox.features_from_place(
    LOCATION,
    tags={
        "amenity": True,
        "shop": True,
        "office": True,
    }
)

# Amenity analysis
gdf_destinations_og = copy.deepcopy(gdf_destinations_raw)
amenity_destinations = gdf_destinations_og["amenity"]
amenity_destinations_counts = amenity_destinations.value_counts()
print("amenity types:")
display(amenity_destinations_counts)

gdf_residential = ox.features_from_place(
    LOCATION,
    tags={"landuse": "residential"}
)

# project to graph CRS
gdf_destinations_raw = gdf_destinations_raw.to_crs(G_master_simplified.graph["crs"])
gdf_residential = gdf_residential.to_crs(G_master_simplified.graph["crs"])

# prepare POI destinations
gdf_destinations_poi = SpatialPrep.prepare_destinations(
    gdf_destinations_raw,
    gdf_local_proj,
)

# clean residential polygons
gdf_residential_clean = SpatialPrep.prepare_residential(gdf_residential)

# attach locality + population
gdf_residential_with_locality = SpatialPrep.attach_locality_and_population_to_residential(
    gdf_residential=gdf_residential_clean,
    gdf_localities=gdf_local_proj,
    locality_to_population=ODConstants.LOCALITY_TO_POPULATION,
)

# build residential destination opportunities for 'visiting'
gdf_residential_destinations = SpatialPrep.build_residential_destinations_from_joined_residential(gdf_residential_with_locality)
# combine POI + residential destinations
gdf_all_destinations = SpatialPrep.append_residential_destinations(gdf_destinations_poi,gdf_residential_destinations,)

print("gdf_all_destinations")
display(gdf_all_destinations)


# build residential origin sampler
residential_sampling = ODSampling.prepare_population_sampling_from_joined_residential(
    gdf_residential_with_locality
)

print("gdf_residential_with_locality")
display(gdf_residential_with_locality)

fig, ax = Plotting.plot_OD_points(
    G_drive,
    gdf_residential=gdf_residential_with_locality,
    gdf_destinations=gdf_all_destinations,
    legend_fontsize=20,
    legend_markerscale=2,
    silhouette=gdf_regions_proj,
    show = False,
    #title="Destination opportunities and residential areas",
)
fig.savefig("Output/Plots/OD/OD_points.pdf", format="pdf", bbox_inches="tight")


Investigating the different amenities and their frequency

In [ ]:
gdf_all_destinations.head()

print("dest_type:")
display(gdf_all_destinations["dest_type"].value_counts())

## OD matrix Generation and Calibration

In [ ]:
purpose_heatmap, region_heatmap = Plotting.plot_od_sampling_heatmaps(
    ODSampling.OUTWARD_PURPOSE_SHARES_BY_DEST_REGION,
    ODSampling.DEST_REGION_GIVEN_ORIGIN,
    show = False,
)

purpose_fig, purpose_ax, purpose_df = purpose_heatmap
region_fig, region_ax, region_df = region_heatmap

purpose_fig.savefig("Output/Plots/OD/purpose_matrix.pdf", format="pdf", bbox_inches="tight")
region_fig.savefig("Output/Plots/OD/region_matrix.pdf", format="pdf", bbox_inches="tight")

In [ ]:
fig, ax = Plotting.plot_population_heatmap_with_regions_and_roads(
    gdf_residential_with_locality,
    gdf_local_proj,
    G=G_drive,
    value_col="raw_weight",
    use_density=False,
    use_log=False,
    cbar_label="Allocated population weight",
    show_roads=True,
    road_edge_color="0.5",      
    road_edge_linewidth=0.5,    
    road_alpha=0.9,
    silhouette=gdf_regions_proj,             
    show_region_fill=True,
    show_region_names=False,
    show_plot=True,             
)
fig.savefig("Output/Plots/OD/Residential_population.pdf", format="pdf", bbox_inches="tight")


OD Calibration

In [ ]:
G_drive = graph_util.make_drive_subgraph(G_master_simplified)
beta_range = np.logspace(-5, -2, 16)
# 1. Get the baseline values
avg_rand_m, abs_rand_avg_m_error, random_od_norm, random_od_counts = ODCalibrator.calculate_random_baseline(
    G=G_drive,
    rng=rng,
    n_trips=1_000,
    locality_to_region=constants.LOCALITY_TO_REGION,
)

# 2. Run your sweep
df_sweep, best_od_norm_matrix, best_od_locality_matrix, best_od_locality_pairs, best_od_locality_timeline_tables, min_abs_avg_m_error, best_avg_m = ODCalibrator.run_beta_sweep(
    G_drive=G_drive,
    residential_sampling=residential_sampling,
    gdf_destinations=gdf_all_destinations,   
    rng=rng,
    beta_range=beta_range,
    list_total_trips_per_hour=OD_GENERATION_TIMELINE,
)

best_od_pairs_list = best_od_locality_pairs

In [ ]:
# 1. Identify the optimal values
min_rmse_idx = df_sweep['abs_avg_m_error'].idxmin()
best_beta = df_sweep.loc[min_rmse_idx, 'beta']
best_len = df_sweep.loc[min_rmse_idx, 'avg_length']

fig,ax = Plotting.plot_od_investigation(
    df_sweep,
    random_avg_length=avg_rand_m,
    random_abs_error=abs_rand_avg_m_error,
    best_beta=best_beta,
    best_avg_m=best_avg_m,
    show_error_diagnostic=False,
)
fig.savefig("Output/Plots/OD/BetaSweep.pdf", format="pdf", bbox_inches="tight")


In [ ]:
paths = paths_util.compute_candidate_paths(G_master_simplified, best_od_pairs_list, "car_cost_current")
edge_importance = paths_util.compute_edge_importance(G_master_simplified, paths, "car_cost_current")

## OD outputing For sumo

In [ ]:
# read network
net = sumolib.net.readNet("../Simulations/Malta_island/Malta_island.net.xml")

gdf_shifted = sumoExporter.sumo_safe_shift_polygons(net, gdf_local_proj_coloured)
sumoExporter.write_taz_polygons(gdf_shifted, "../Simulations/localities.poly.xml")
#display(gdf_local_proj_coloured)

In [ ]:
sumo_export_rng = np.random.default_rng(SEED)

sumo_od_timeline = ODGeneration.gen_od_trips_timeline(
    SUMO_OD_GENERATION_TIMELINE,
    G_drive,
    residential_sampling,
    gdf_all_destinations,
    sumo_export_rng,
    beta=best_beta,
)

sumo_od_pairs = ODAggregation.aggregate_timeline_ods(sumo_od_timeline)
sumo_od_locality_timeline_tables = ODAggregation.build_region_od_tables_from_timeline(
    sumo_od_timeline,
    G_drive,
)
sumo_od_locality_timeline_tables = sumoExporter.make_sumo_safe_od_timeline_tables(
    sumo_od_locality_timeline_tables
)

display(sumo_od_locality_timeline_tables)

sumoExporter.write_od_matrix(sumo_od_locality_timeline_tables, "../Simulations/od_matrix.xml")
print(f"Exported {sum(SUMO_OD_GENERATION_TIMELINE):,} SUMO OD trips with beta={best_beta:g}")

## OD Pathing

In [ ]:
fig, ax, _ = Plotting.plot_OD_lines(
    G_plotting,
    best_od_locality_pairs,
    alpha=0.1,
    linewidth=1.5,
    #title="OD Straight-Line Connections",
    silhouette=gdf_local_proj,
    show = False,
)
fig.savefig("Output/Plots/OD/OD_lines.pdf", format="pdf", bbox_inches="tight")

# Optimisation Approaches 

## Heuristic Approaches

In [ ]:
num_nodes = len(G_master_simplified.nodes)
print(num_nodes)
investigation.compare_two_k(G_master_simplified, k1=num_nodes, k2=50, seed=12)

In [ ]:
G_bike_full = graph_util.make_bikeable_subgraph(G_master_simplified)
OD_pairs = best_od_pairs_list

bike_nodes = set(G_bike_full.nodes())
matched = sum(1 for (o, d) in OD_pairs if int(o) in bike_nodes and int(d) in bike_nodes)
print(f"{matched} / {len(OD_pairs)} OD pairs have both endpoints in G_bike_full")

sample_edges = list(G_bike_full.edges(data=True))[:3]
for u, v, d in sample_edges:
    print(d.get("bike_cost_penalty"), d.get("length"))

print(list(OD_pairs.items())[:5])

In [ ]:
heuristics = {
    "edge_betweenness": heuristic_optim.heuristic_segment_betweenness_centrality,
    "L2C": heuristic_optim.heuristic_L2C,
    "OD_betweenness": heuristic_optim.heuristic_od_segment_betweenness,
    "Random": heuristic_optim.heuristic_random,
}

#NOTE majority of computation time is spent on evaluation 
all_results = []
optimised_graphs = {}
for name, func in tqdm(heuristics.items(), desc="Evaluating Heuristics"):
    tqdm.write(f"Running {name} heuristic")

    optimised_graph, evaluations_df = graph_optimisation_heuristic.run_optimisation(
        G_master_simplified,
        heuristic_func=func,
        od = best_od_pairs_list,
        n_iterations=1_500,
        EVALUATION_MOD=100,
        k_sample = 50, 
        dry_run=False,
        heuristic_name=name,
    )
    evaluations_df["heuristic"] = name
    all_results.append(evaluations_df)

    optimised_graphs[name] = optimised_graph

    diff_log_series = evaluations_df["diff_log"]
    diff_log = diff_log_series.explode().dropna().tolist()

    evaluations_df.to_csv(f'{name}_eval_df.csv')
    
    with open(f"Output/ProposedGraphs/{name}_diff_log.json", "w") as f:
        json.dump(diff_log, f)

    with open(f"Output/ProposedGraphs/{name}_graph.pkl", "wb") as f:
        pickle.dump(optimised_graph, f)


optimised_graph_results_df = pd.concat(all_results, ignore_index=True)
optimised_graph_results_df.sort_values(by=["heuristic", "iteration"])
optimised_graph_results_df.head()

optimised_graph_results_df.to_csv(f'Output/ProposedGraphs/optimised_graph_results_df.csv')

In [ ]:
optimised_graph_results_df = pd.read_csv('Output/ProposedGraphs/optimised_graph_results_df.csv')

In [ ]:
stop_rows = []

for heuristic_name, df in optimised_graph_results_df.sort_values("iteration").groupby("heuristic"):
    recorded_stops = df[
        df["stop_reason"].notna()
        & (df["stop_reason"].astype(str).str.len() > 0)
    ]

    if len(recorded_stops):
        row = recorded_stops.iloc[0]
        reason = row["stop_reason"]
    else:
        row = df.iloc[-1]
        reason = "no recorded stop condition"

    stop_rows.append({
        "heuristic": heuristic_name,
        "iteration": int(row["iteration"]),
        "stop_reason": reason,
        "bike_gain_vs_baseline": row.get("bike_gain_vs_baseline"),
        "car_harm_vs_baseline": row.get("car_harm_vs_baseline"),
    })

stop_conditions_df = pd.DataFrame(stop_rows)
display(stop_conditions_df)

for _, row in stop_conditions_df.iterrows():
    print(f"{row['heuristic']}: {row['stop_reason']} at iteration {row['iteration']}")


In [ ]:
Plotting.plot_metrics(optimised_graph_results_df)

In [ ]:
for heuristic_name, _ in heuristics.items():

    with open(f"Output/ProposedGraphs/{heuristic_name}_graph.pkl", "rb") as f:
        G_optimised = pickle.load(f)


    fig, ax = Plotting.plot_proposed_cycling_network(
        G_master_simplified,
        G_optimised,
        initial_safety_classes=(constants.SafetyClass.PROTECTED,),
        #title = f"{heuristic_name} Proposed Cycling Network",
        fig_size=(20, 20),
        show = False,
    )

    fig.savefig(f"Output/Plots/ProposedGraphs/{heuristic_name}.pdf", format="pdf", bbox_inches='tight')

# Sumo

## Sumo Mapping

In [ ]:
optimised_graph_results_df = pd.read_csv("Output/ProposedGraphs/optimised_graph_results_df.csv")
for heuristic_name, _ in heuristics.items():
    optimised_chosen_heuristic_df = optimised_graph_results_df[
        optimised_graph_results_df["heuristic"] == heuristic_name
    ].sort_values("iteration")

    diff_log = []

    for value in optimised_chosen_heuristic_df["diff_log"].dropna():
        if isinstance(value, str):
            events = ast.literal_eval(value)
        else:
            events = value

        if isinstance(events, list):
            diff_log.extend(events)

    print("events loaded:", len(diff_log))
    print(pd.Series([e.get("event_type") for e in diff_log if isinstance(e, dict)]).value_counts())

    G_optimised, projection_stats = graph_util.apply_reallocation_events_to_unsimplified(
        G_master_simplified,
        G_master_unsimplified_oneway,
        diff_log,
        return_stats=True,
    )

    print(heuristic_name, projection_stats)

    enrich_attributes.bike_safety_classification(G_optimised)
    impedance_calculator.update_bike_costs(G_optimised)

    ox.save_graph_xml(
        G_optimised,
        f"../Simulations/{heuristic_name}/{heuristic_name}.osm",
    )


In [ ]:
fig, ax = Plotting.plot_categorical_attr(G_optimised, "safety", constants.SAFETY_COLORS, title = "Malta UNSIMPLIFIED SUMO Car Road Safety Classification", fig_size=(10,10), dpi=500)
fig.savefig("Malta_Road_Safety_Classification.svg", format="svg", bbox_inches="tight")

## SUMO run

In [ ]:
simulation_str = "Malta_island"
sumoRun.run_simulation(simulation_str, car_scale=0.993, bike_scale=0.007,
                       should_build_network = False,  force_taz_update = False, should_gen_trips = False, should_run_router=False, should_execute_sim = True, verbose =True)

for simulation_str, _ in heuristics.items():
    sumoRun.run_simulation(simulation_str, car_scale=0.9, bike_scale=0.1,
                            should_build_network = False,  force_taz_update = False, should_gen_trips = False, should_run_router=False, should_execute_sim = True, verbose =True)

## Sumo Analysis

General Overall Inpection

In [ ]:
simulation_names = list(dict.fromkeys([
    *heuristics.keys(),
    "Malta_island",
]))

sumo_simulation_results = {}

for heuristic_name in simulation_names:
    output_folder = f"../Simulations/{heuristic_name}/Output"

    try:
        summary_df = sumo_util.load_summary(f"{output_folder}/summary.xml")
        car_df = sumo_util.load_aggregated_meandata(
            f"{output_folder}/network_car_meandata.xml",
            mode="car",
        )
        bike_df = sumo_util.load_aggregated_meandata(
            f"{output_folder}/network_bike_meandata.xml",
            mode="bike",
        )
        tripinfo_df = sumo_util.load_tripinfo(f"{output_folder}/tripinfo.xml")

    except (FileNotFoundError, ET.ParseError) as exc:
        print(f"Skipping {heuristic_name}: {exc}")
        continue

    sumo_simulation_results[heuristic_name] = {
        "summary": summary_df,
        "car": car_df,
        "bike": bike_df,
        "tripinfo": tripinfo_df,
    }

In [ ]:
Plotting.plot_compare_summary( sumo_simulation_results, metric="running", ylabel="Vehicles", title="Vehicles Running in Network",)

Plotting.plot_compare_summary( sumo_simulation_results, metric="halting", ylabel="Vehicles", title="Vehicles halted in Network",)

Plotting.plot_compare_summary( sumo_simulation_results, metric="meanWaitingTime", scale=1, ylabel="Average total waiting Time (seconds)", title="Average waiting Time",)

Plotting.plot_compare_summary( sumo_simulation_results, metric="meanTravelTime", ylabel="Mean Travel Time (minutes)", title="Mean Travel Time", scale=(1/60),)

Plotting.plot_compare_summary( sumo_simulation_results, metric="meanSpeedRelative", ylabel="meanSpeedRelative", title="meanSpeedRelative",)



Plotting.plot_compare_mode( sumo_simulation_results, mode="car", metric="speed", scale=3.6, ylabel="Speed (km/h)", title="Mean Car Speed",)

Plotting.plot_compare_mode( sumo_simulation_results, mode="bike", metric="speed", scale=3.6, ylabel="Speed (km/h)", title="Mean Bike Speed",)